# Screen Macro — Object Detection Training

Trains a small object-detection model on a dataset captured/labeled by Screen Macro's "Train new model…" wizard (Detect Object action), and exports it to the `.onnx` format the app's `onnx_detector.py` expects.

**Colab (default, manual):**
1. `Runtime` → `Change runtime type` → select a **GPU** (T4 is fine, free tier).
2. `Runtime` → `Run all`.
3. Wait for a "Choose files" button to appear below step 2's cell (can take a minute — it installs dependencies first), click it, and upload the `..._dataset.zip` file Screen Macro produced.
4. Everything else runs on its own — synthetic-frame generation, training, export, and an optional sanity check (step 7, safe to skip) — then `best.onnx` downloads automatically at the end. Import it back into the same action via "Import model…", and set its **class filter to 0** (the model's other output class, `1`, is background — see step 6's note).

**Kaggle (opt-in, scripted):** the app's `cv_training.py` pushes this notebook as a Kaggle kernel with the dataset attached as a Kaggle Dataset, polls until it finishes, and pulls `best.onnx` back automatically — no manual steps. The notebook auto-detects which platform it's running on (`ON_KAGGLE`, cell 4) and adjusts the upload/download cells accordingly; everything else runs identically either way.

**Status: verified end-to-end on a real Colab run (2026-08-21)** — training a keypoint-preview model through all 50 epochs and exporting a working `best.onnx` that produces sane detections. The export step (5) needed substantially more rework than originally expected; see its own cell for what changed and why.


## 1. Install dependencies

Pinned, not "latest" — several real breakages this notebook hit along the way (a `torch.onnx.export` default flip, a missing `pytorch_lightning` extra, an internal keypoint-schema quirk) all came from an upstream release silently changing behavior underneath an unpinned install. These versions are what every other cell in this notebook has actually been verified against. Worth revisiting occasionally (bump the pins, re-run end-to-end, confirm nothing broke) to pick up real upstream fixes -- but that should be a deliberate, tested decision, not something that happens by default on every fresh Colab run.

In [ ]:
!pip install -q "rfdetr[train,loggers,augment]==1.9.4" "onnx==1.22.0" "onnxruntime==1.29.0"

## 2. Upload and unpack the dataset

Expects the zip Screen Macro's "Train new model…" wizard produces: an `images/` folder of captured frames plus a `labels.json` of the form
`{"images_dir": "images", "labels": [{"image": "frame_001.png", "objects": [{"box": [x, y, w, h], "keypoint": [x, y]}, ...]}, ...]}`
(frames with no entry in `labels` were skipped during labeling — the object wasn't visible in them; each frame can list more than one object).


In [ ]:
import os, glob, zipfile, shutil
from pathlib import Path

RAW_DIR = Path("dataset_raw")
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

# cv_training.py's push already uploaded the dataset zip as this kernel's one
# attached Kaggle Dataset -- mounted read-only under /kaggle/input, no interactive
# upload prompt needed the way Colab's files.upload() requires. Detected by whether
# a zip is actually there, not just os.path.exists("/kaggle/input") -- that directory
# exists (empty) on plain Colab too, which previously made this always take the
# Kaggle branch there and crash on an empty candidates list.
candidates = glob.glob("/kaggle/input/*/*.zip") + glob.glob("/kaggle/input/*.zip")
ON_KAGGLE = bool(candidates)
if ON_KAGGLE:
    zip_name = candidates[0]
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(RAW_DIR)

print("Extracted:", list(RAW_DIR.iterdir()))


## 3. Synthesize additional training frames (copy-paste augmentation)

Real captured datasets from Screen Macro's "Train new model…" wizard are usually just a few dozen frames — this step multiplies that for free by cropping each labeled object out of its source frame, randomly rotating/zooming it, and compositing it onto a *different* frame already captured for this same dataset (with a softly feathered edge, not a hard rectangle). This is "copy-paste augmentation," a well-established object-detection technique, not something novel/unproven — it doesn't require clean/empty backgrounds or segmentation masks, just the boxes/keypoints already labeled.

`SYNTHETIC_MULTIPLIER` controls how many synthetic variants get generated per real labeled object. Synthetic entries are tagged `"synthetic": true` in `labels.json` so the next step's train/valid split (step 4) can keep them out of `valid` entirely — evaluation always runs against real, unmodified captures only.


In [ ]:
import json, random
import cv2
import numpy as np

SYNTHETIC_MULTIPLIER = 3    # extra composited variants generated per real labeled object
ROTATION_RANGE_DEG = 20     # +/- degrees applied to the cropped object before compositing
SCALE_RANGE = (0.7, 1.3)    # zoom out/in applied to the cropped object
CROP_MARGIN = 0.15          # extra margin around the object's box when cropping, as a fraction of box size
FEATHER_PX = 8              # soft-edge width, in pixels, blended at the pasted crop's border

random.seed(0)  # reproducible synthesis, matching the train/valid split's own random.seed(0) next step

with open(RAW_DIR / "labels.json") as f:
    raw = json.load(f)
images_dir_name = raw.get("images_dir", "images")
images_dir = RAW_DIR / images_dir_name
real_labels = raw["labels"]
all_frame_names = sorted(p.name for p in images_dir.glob("*.png"))


def _rotate_scale_crop(crop_bgr, keypoint_local, angle_deg, scale):
    """Rotates+scales a crop (and its local keypoint coords) around its own center,
    expanding the canvas so nothing gets clipped. Returns (rotated_bgr, alpha_mask,
    new_keypoint_local) -- alpha_mask is a feathered soft-edge mask for blending, not a
    hard rectangle."""
    h, w = crop_bgr.shape[:2]
    diag = int(np.ceil(np.hypot(h, w) * max(scale, 1.0))) + 2 * FEATHER_PX
    cx, cy = w / 2, h / 2
    m = cv2.getRotationMatrix2D((cx, cy), angle_deg, scale)
    m[0, 2] += diag / 2 - cx
    m[1, 2] += diag / 2 - cy
    rotated = cv2.warpAffine(crop_bgr, m, (diag, diag), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    alpha = np.full((h, w), 255, dtype=np.uint8)
    alpha_warped = cv2.warpAffine(alpha, m, (diag, diag), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    if FEATHER_PX > 0:
        alpha_warped = cv2.GaussianBlur(alpha_warped, (0, 0), sigmaX=FEATHER_PX / 2)
    kx, ky = keypoint_local
    pt = m @ np.array([kx, ky, 1.0])
    return rotated, alpha_warped, (float(pt[0]), float(pt[1]))


def _composite(background_bgr, patch_bgr, patch_alpha, top_left_xy):
    """Alpha-blends patch_bgr onto background_bgr at top_left_xy in place, clipped to
    bounds. Returns whether anything was actually blended (False if fully off-frame)."""
    bx, by = top_left_xy
    bh, bw = background_bgr.shape[:2]
    ph, pw = patch_bgr.shape[:2]
    x0, y0 = max(bx, 0), max(by, 0)
    x1, y1 = min(bx + pw, bw), min(by + ph, bh)
    if x1 <= x0 or y1 <= y0:
        return False
    px0, py0 = x0 - bx, y0 - by
    px1, py1 = px0 + (x1 - x0), py0 + (y1 - y0)
    region = background_bgr[y0:y1, x0:x1].astype(np.float32)
    patch_region = patch_bgr[py0:py1, px0:px1].astype(np.float32)
    alpha_region = (patch_alpha[py0:py1, px0:px1].astype(np.float32) / 255.0)[..., None]
    background_bgr[y0:y1, x0:x1] = (region * (1 - alpha_region) + patch_region * alpha_region).astype(np.uint8)
    return True


synthetic_labels = []
synthetic_count = 0
for entry in real_labels:  # only ever source from the REAL entries -- synthetic ones never feed back in
    src_img = cv2.imread(str(images_dir / entry["image"]))
    if src_img is None:
        continue
    for obj in entry["objects"]:
        bx, by, bw, bh = obj["box"]
        kx, ky = obj["keypoint"]
        mx, my = int(bw * CROP_MARGIN), int(bh * CROP_MARGIN)
        cx0, cy0 = max(0, bx - mx), max(0, by - my)
        cx1, cy1 = min(src_img.shape[1], bx + bw + mx), min(src_img.shape[0], by + bh + my)
        crop = src_img[cy0:cy1, cx0:cx1]
        if crop.size == 0:
            continue
        local_kx, local_ky = kx - cx0, ky - cy0

        # Prefer a genuinely different background frame; fall back to this same frame
        # (still adds pose/rotation/scale variety even without new background context)
        # when the dataset is too small to have another one to pick from.
        candidate_bgs = [n for n in all_frame_names if n != entry["image"]] or all_frame_names

        for _ in range(SYNTHETIC_MULTIPLIER):
            bg_name = random.choice(candidate_bgs)
            bg_img = cv2.imread(str(images_dir / bg_name))
            if bg_img is None:
                continue
            bg_img = bg_img.copy()
            angle = random.uniform(-ROTATION_RANGE_DEG, ROTATION_RANGE_DEG)
            scale = random.uniform(*SCALE_RANGE)
            patch, alpha, (new_kx, new_ky) = _rotate_scale_crop(crop, (local_kx, local_ky), angle, scale)
            ph, pw = patch.shape[:2]
            bh_img, bw_img = bg_img.shape[:2]
            if pw >= bw_img or ph >= bh_img:
                continue  # patch too big for this background at this scale -- skip this variant
            top_left = (random.randint(0, bw_img - pw), random.randint(0, bh_img - ph))
            if not _composite(bg_img, patch, alpha, top_left):
                continue

            # New box: the warped alpha mask's own bounding box tracks the rotated
            # content's true footprint -- rotating the original box's corners instead
            # would over/under-estimate a rotated rectangle's actual extent.
            ys, xs = np.where(alpha > 32)
            if len(xs) == 0:
                continue
            new_box = [
                int(top_left[0] + xs.min()), int(top_left[1] + ys.min()),
                int(xs.max() - xs.min()), int(ys.max() - ys.min()),
            ]
            new_keypoint = [round(top_left[0] + new_kx), round(top_left[1] + new_ky)]

            synthetic_count += 1
            synth_name = f"synth_{synthetic_count:05d}.png"
            cv2.imwrite(str(images_dir / synth_name), bg_img)
            synthetic_labels.append({
                "image": synth_name,
                "objects": [{"box": new_box, "keypoint": new_keypoint}],
                "synthetic": True,  # keeps this out of the valid split in the next step
            })

with open(RAW_DIR / "labels.json", "w") as f:
    json.dump({"images_dir": images_dir_name, "labels": real_labels + synthetic_labels}, f, indent=2)

n_real_objects = sum(len(e["objects"]) for e in real_labels)
print(
    f"Added {len(synthetic_labels)} synthetic composited frame(s) from {n_real_objects} real "
    f"labeled object(s) across {len(real_labels)} real frame(s) -- "
    f"{len(real_labels) + len(synthetic_labels)} total labeled frames before the train/valid split."
)

## 4. Convert to COCO format (train/valid split)

One category (`"object"`) with one keypoint (`"click_point"`) per COCO's keypoint-annotation convention — `keypoints: [x, y, visibility]`, `visibility=2` meaning "labeled and visible" (every entry here is, since the labeling step only records a keypoint when the object was actually clicked on). A 90/10 split is a reasonable default for a small, single-object dataset; adjust `VALID_FRACTION` if you captured a lot more images. Synthetic entries from step 3 are excluded from this split entirely -- they're appended onto `train` afterward, never `valid`.

In [ ]:
import json, random

VALID_FRACTION = 0.1
COCO_DIR = Path("dataset_coco")

with open(RAW_DIR / "labels.json") as f:
    raw = json.load(f)
images_dir = RAW_DIR / raw.get("images_dir", "images")
entries = raw["labels"]

# Synthetic (copy-paste-composited, see step 3) entries never enter "valid" -- the split
# below is computed from the real entries only, then every synthetic entry is appended
# straight onto "train" so evaluation always runs against real, unmodified captures.
real_entries = [e for e in entries if not e.get("synthetic")]
synthetic_entries = [e for e in entries if e.get("synthetic")]
random.seed(0)
random.shuffle(real_entries)
n_valid = max(1, int(len(real_entries) * VALID_FRACTION)) if len(real_entries) > 1 else 0
splits = {"valid": real_entries[:n_valid], "train": real_entries[n_valid:] + synthetic_entries}

import cv2

for split_name, split_entries in splits.items():
    split_dir = COCO_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    images_out, annotations_out = [], []
    for i, e in enumerate(split_entries, start=1):
        src = images_dir / e["image"]
        img = cv2.imread(str(src))
        h, w = img.shape[:2]
        shutil.copy(src, split_dir / e["image"])
        images_out.append({"id": i, "file_name": e["image"], "width": w, "height": h})
        # Each frame can carry more than one labeled object (see the schema note
        # above) -- one COCO annotation per object, all sharing this frame's image_id.
        for obj in e["objects"]:
            bx, by, bw, bh = obj["box"]
            kx, ky = obj["keypoint"]
            annotations_out.append({
                "id": len(annotations_out) + 1, "image_id": i, "category_id": 1,
                "bbox": [bx, by, bw, bh],
                "area": bw * bh,
                "iscrowd": 0,
                "keypoints": [kx, ky, 2],
                "num_keypoints": 1,
            })
    coco = {
        "images": images_out,
        "annotations": annotations_out,
        "categories": [{
            "id": 1, "name": "object", "supercategory": "object",
            "keypoints": ["click_point"], "skeleton": [],
        }],
    }
    with open(split_dir / "_annotations.coco.json", "w") as f:
        json.dump(coco, f)
    print(f"{split_name}: {len(images_out)} image(s), {len(annotations_out)} annotation(s)")

## 5. Train

Tries RF-DETR's keypoint-preview model first (predicts the click point directly, per `docs/cv-object-detection-investigation.md`'s recommendation); falls back to the plain box-only model if the preview class isn't importable or training on it fails — a fully-supported degradation, not an error path (`onnx_detector.py` already handles a keypoint-less `[N,6]` export by clicking the box center instead).

**Augmentation:** rfdetr's default (no `aug_config` passed) only ever applies `RandomHorizontalFlip`, which is auto-disabled anyway for this keypoint schema (no left/right flip pairs configured) — so without the custom `AUG_CONFIG` below, training would run with effectively *no* augmentation on a dataset that's usually just a few dozen real captured frames. Mild brightness/contrast, a small rotation, and light blur/noise (mirroring rfdetr's own "industrial" preset, meant for lighting/sensor noise — a reasonable stand-in for video-encoding artifacts and inconsistent in-game lighting) multiply that small dataset's effective diversity for free. Deliberately no flips or heavy rotation: game UI elements are normally upright and axis-aligned, so augmenting toward flipped/rotated variants would fight the real data instead of reflecting it. Requires the `rfdetr[augment]` extra (already included in step 1's install).

**Per-epoch metrics** (mAP/mAR/F1) print below this cell as each epoch finishes, and are also logged to TensorBoard — run this in a separate cell (before or after starting training) to watch them update live:
```python
%load_ext tensorboard
%tensorboard --logdir output
```

**Early stopping** is enabled below: training stops once the monitored metric (mAP@50:95) hasn't improved by at least `EARLY_STOPPING_MIN_DELTA` for `EARLY_STOPPING_PATIENCE` epochs in a row, instead of always running the full `EPOCHS`. Raise the patience (or set `early_stopping=False` in both `model.train()` calls below) if a small/noisy dataset stops too eagerly.

**mAP-ceiling shortcut:** mAP@50:95 is bounded at 1.0 — once both the regular and EMA tracks reach it, neither can ever register a further "improvement," so ordinary early stopping would still wait out the full `EARLY_STOPPING_PATIENCE` epochs confirming that before stopping. A small custom callback below stops immediately instead once both hit the ceiling. Reaching 1.0 usually just means every image in a small validation split scored perfectly, not that the model has learned everything it could ever need — worth still checking the exported model against real, unseen game screens via the app's own Detect Model Validate regardless of how training stopped.


In [ ]:
KEYPOINT_MODE = True
try:
    from rfdetr import RFDETRKeypointPreview
    model = RFDETRKeypointPreview()
except Exception as exc:
    print("Keypoint-preview model unavailable, falling back to box-only:", exc)
    KEYPOINT_MODE = False

if not KEYPOINT_MODE:
    from rfdetr import RFDETRNano
    model = RFDETRNano()

EPOCHS = 200      # small dataset -- more epochs than a COCO-scale run, adjust if it overfits
BATCH_SIZE = 4    # keep small: free-tier T4 memory headroom is the main constraint here
EARLY_STOPPING_PATIENCE = 20     # stop once mAP@50:95 hasn't improved for this many epochs
EARLY_STOPPING_MIN_DELTA = 0.001 # smallest mAP improvement that still counts as progress

# rfdetr's own default (aug_config left unset) only ever applies RandomHorizontalFlip --
# and that gets silently disabled for this keypoint schema anyway (no left/right flip
# pairs configured for a single "click_point" keypoint), so training would otherwise run
# with effectively NO augmentation at all on a dataset that's usually just a few dozen
# real captured frames. This conservative custom config (mild brightness/contrast, a
# small rotation, light blur/noise) multiplies that small dataset's effective diversity
# for free -- probabilities/ranges kept small since aggressive augmentation on a tiny
# dataset can hurt more than help (see rfdetr's own aug_configs.py docstring). Blur/noise
# specifically mirror rfdetr's own "AUG_INDUSTRIAL" preset (built for lighting/sensor
# noise), a reasonable stand-in for video-encoding artifacts and inconsistent in-game
# lighting. No (Horizontal/Vertical)Flip or heavy rotation: game UI elements are normally
# axis-aligned and upright, so anything that teaches the model to expect a flipped/rotated
# button would fight against, not reflect, the real variation in this data. Requires the
# `rfdetr[augment]` extra (installed in step 1) -- without it, passing a non-empty
# aug_config raises inside model.train() when it tries to build the Albumentations
# pipeline.
AUG_CONFIG = {
    "RandomBrightnessContrast": {"brightness_limit": 0.15, "contrast_limit": 0.15, "p": 0.4},
    "Rotate": {"limit": 10, "p": 0.3},
    "GaussianBlur": {"blur_limit": 3, "p": 0.25},
    "GaussNoise": {"std_range": (0.01, 0.04), "p": 0.25},
}

# Optional: stop the instant both the regular and EMA tracks hit mAP@50:95's 1.0 ceiling,
# instead of still waiting out the full EARLY_STOPPING_PATIENCE window afterward -- neither
# can ever register a further "improvement" past 1.0, so those extra epochs are pure waste.
# rfdetr's own docs recommend extending trainer.callbacks right after build_trainer()
# returns as the supported way to add custom callbacks; model.train() calls build_trainer()
# internally though, so this wraps it to inject the extra callback the same way,
# transparently.
#
# Two real bugs fixed here, both only surfacing on a SECOND run of this cell in the same
# Colab session (re-running after an error, or "Run all" a second time without a runtime
# restart -- module-level monkeypatches like this persist across cell re-runs):
# 1. `_orig_build_trainer` was a plain global name, not a value bound at function-definition
#    time -- every wrapper's body looked up whatever that name CURRENTLY resolved to at
#    CALL time, not what it was when that particular wrapper was defined. Re-running this
#    cell reassigns the same global, so the previous run's wrapper ends up calling itself
#    -- infinite recursion (RecursionError: maximum recursion depth exceeded), exactly what
#    happened here on both the keypoint attempt and its box-only fallback. Fixed by binding
#    the original via a default argument (`_orig=...`), which Python evaluates once at `def`
#    time and never re-reads afterward -- immune to the enclosing global changing later.
# 2. Even with that fixed, re-running this cell would still stack a new wrapper on top of
#    the previous run's wrapper every time (harmless but wasteful -- the callback would get
#    appended more than once). Fixed with an `_ceiling_stop_installed` marker attribute so
#    installing the hook a second time in the same session is a clean no-op.
#
# Still wrapped in try/except: any failure here just logs a warning and leaves ordinary
# patience-based early stopping in place -- it never blocks training.
try:
    import pytorch_lightning as pl
    import rfdetr.training as _rfdetr_training

    class _StopAtMapCeiling(pl.Callback):
        """Stops training once val/mAP_50_95 and val/ema_mAP_50_95 both reach ~1.0 -- the
        max possible mAP@50:95 score. Note this usually means every validation image
        scored perfectly on a small validation split, not that the model has learned
        everything it could ever need -- see docs/cv-object-detection-investigation.md
        for the "is 1.0 really 'done'?" discussion this was added after."""

        _CEILING = 0.9999  # allow for float rounding just under the true 1.0 max

        def on_validation_epoch_end(self, trainer, pl_module):
            metrics = trainer.callback_metrics
            regular = metrics.get("val/mAP_50_95")
            ema = metrics.get("val/ema_mAP_50_95")
            if regular is None or ema is None:
                return
            if float(regular) >= self._CEILING and float(ema) >= self._CEILING:
                print(
                    f"Both regular ({float(regular):.4f}) and EMA ({float(ema):.4f}) mAP@50:95 "
                    "hit the ceiling -- stopping now instead of waiting out early stopping's patience."
                )
                trainer.should_stop = True

    if not getattr(_rfdetr_training.build_trainer, "_ceiling_stop_installed", False):
        def _build_trainer_with_ceiling_stop(*args, _orig=_rfdetr_training.build_trainer, **kwargs):
            trainer = _orig(*args, **kwargs)
            trainer.callbacks.append(_StopAtMapCeiling())
            return trainer

        _build_trainer_with_ceiling_stop._ceiling_stop_installed = True
        _rfdetr_training.build_trainer = _build_trainer_with_ceiling_stop
except Exception as exc:
    print(f"Could not enable the mAP-ceiling stop shortcut ({exc}) -- early stopping will still work normally.")

try:
    model.train(
        dataset_dir=str(COCO_DIR), epochs=EPOCHS, batch_size=BATCH_SIZE,
        early_stopping=True, early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, aug_config=AUG_CONFIG,
    )
except Exception as exc:
    if KEYPOINT_MODE:
        print("Keypoint-preview training failed, falling back to box-only:", exc)
        from rfdetr import RFDETRNano
        KEYPOINT_MODE = False
        model = RFDETRNano()
        model.train(
            dataset_dir=str(COCO_DIR), epochs=EPOCHS, batch_size=BATCH_SIZE,
            early_stopping=True, early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, aug_config=AUG_CONFIG,
        )
    else:
        raise

## 6. Export to ONNX with the app's expected contract

`onnx_detector.py` expects a single output tensor per image: `[N, 6]` (x1, y1, x2, y2, confidence, class_id) or `[N, 8]` (…+keypoint_x, keypoint_y) in the model's own fixed input-size pixel coordinates — with NMS/decoding already applied, so the shipped app never needs architecture-specific postprocessing.

**Syncs the keypoint schema before tracing** — `model.model.postprocess` is built once at model construction time, before the pretrained checkpoint's own keypoint schema even loads, let alone this dataset's real one being inferred. `model.train()` trains a separate, correctly-schema'd model+postprocess internally and only syncs the raw weights back onto `model.model` afterward, leaving `model.model.postprocess` stuck on the pretrained checkpoint's own schema (2 classes × 17 keypoints for RF-DETR's stock keypoint-preview checkpoint — the standard COCO-pose layout) instead of this dataset's real one (typically 1 keypoint, single class, for this app's single "click_point" per object). Left unsynced, tracing crashes with `PostProcess._decode_keypoints_for_image`'s `"keypoints_i padded slot dimension (N) must equal num_keypoint_classes (...) * max_num_keypoints (...)"`.

**Loads the best checkpoint (`output/checkpoint_best_total.pth`) before tracing** — `model.train()` only leaves the last trained epoch's weights in memory, not necessarily the best-scoring one. With early stopping enabled (see step 5), training deliberately keeps running for `EARLY_STOPPING_PATIENCE` epochs past the actual best before stopping, so exporting the in-memory weights as-is would ship a worse model than what's already sitting on disk. Falls back to the in-memory weights (with a printed warning) if that checkpoint file isn't found for any reason.

This traces the model's own export-mode forward pass (`raw_module.export()` + `forward_export()`) together with its `PostProcess` module directly — both are plain tensor-op code, unlike `predict()`'s heavy multi-input-type handling and `Detections`/`KeyPoints` object construction, which turned out to be untraceable (and, when the raw training-mode forward was traced directly without calling `.export()` first, crashed with a CUDA illegal memory access — multi-scale deformable attention's custom CUDA kernel isn't traceable at all without that step).

Two of `rfdetr`'s own postprocessing internals also needed patching around for export specifically — both scoped to this one `PostProcess` instance, not the whole class, so nothing outside this cell is affected:
- `PostProcess._select_topk` uses `torch.argsort(..., stable=True)`, which the legacy ONNX exporter can't convert at all (`Sort, Out parameter is not supported` — a real version mismatch inside PyTorch itself, not an `rfdetr` bug). Replaced with an equivalent `torch.topk`-based version for the export trace only — same descending-score order, just without the exactly-tied-score tie-break rule, which doesn't matter for this app's use.
- Keypoint-uncertainty score fusion (`trace_alpha`) calls `torch.nextafter`, also unsupported by ONNX. Disabled for export since the app only needs the plain confidence score, not the uncertainty-weighted refinement.

**The exported model's output mixes foreground and background rows** (`class_id=0` is foreground, `class_id=1` is background, per this one-class keypoint setup) — set the Detect Model action's class filter to `0` in the app so it never treats a background row as the best match.


In [ ]:
import torch
import torch.nn as nn
import types

INPUT_SIZE = model.model.resolution

class ExportWrapper(nn.Module):
    """Wraps the raw export-mode LWDETR forward + PostProcess.forward() -- both plain
    tensor ops, no image-decoding/type-dispatch/numpy machinery -- into one flat
    [N,6]/[N,8] tensor per image. Reimplements just the tensor-math half of what
    predict() does; skips predict()'s heavy multi-input-type handling and
    Detections/KeyPoints object construction entirely, which is what made tracing
    predict() itself fail. raw_module.export() must already have been called before
    this wraps it (swaps in export-safe submodule implementations, e.g. multi-scale
    deformable attention's CUDA kernel -- tracing it directly without this step
    crashes with a CUDA illegal memory access)."""

    def __init__(self, raw_module, postprocess_module, keypoint_mode: bool, input_size: int):
        super().__init__()
        self.raw_module = raw_module
        self.postprocess = postprocess_module
        self.keypoint_mode = keypoint_mode
        self.input_size = input_size

    def forward(self, x):
        outputs_coord, outputs_class, *rest = self.raw_module(x)
        out_dict = {"pred_logits": outputs_class, "pred_boxes": outputs_coord}
        if self.keypoint_mode and rest:
            out_dict["pred_keypoints"] = rest[0]
        target_sizes = torch.tensor([[self.input_size, self.input_size]], device=x.device)
        result = self.postprocess(out_dict, target_sizes)[0]  # batch size 1 -> one dict
        boxes = result["boxes"]                     # (num_select, 4) absolute xyxy
        scores = result["scores"].unsqueeze(-1)
        labels = result["labels"].unsqueeze(-1).float()
        cols = [boxes, scores, labels]
        if self.keypoint_mode and "keypoints" in result:
            cols.append(result["keypoints"][:, 0, :2])  # our one "click_point" keypoint's (x, y)
        return torch.cat(cols, dim=-1)


def _select_topk_onnx_friendly(self, out_logits):
    """Replaces PostProcess._select_topk for export only -- see this cell's markdown
    for why its own torch.argsort(..., stable=True) can't be exported."""
    prob = out_logits.sigmoid()
    logits_for_topk = prob.view(out_logits.shape[0], -1)
    num_to_select = min(self.num_select, logits_for_topk.shape[1])
    topk_values, topk_indexes = torch.topk(logits_for_topk, num_to_select, dim=1)
    scores = topk_values
    topk_boxes = topk_indexes // out_logits.shape[2]
    labels = topk_indexes % out_logits.shape[2]
    return scores, labels, topk_boxes


raw_module = model.model.model

# model.model.postprocess was built once when RFDETRKeypointPreview()/RFDETRNano() was
# first constructed -- before the pretrained checkpoint's own keypoint schema even
# loaded, let alone this dataset's real schema being inferred. model.train() reconfigures
# a SEPARATE, freshly-built model+postprocess pair (inside RFDETRModelModule) to the
# dataset's real schema and trains that one -- then only syncs the raw WEIGHTS back onto
# model.model, never model.model.postprocess itself. So model.model.postprocess is left
# holding the pretrained checkpoint's own schema (for RFDETRKeypointPreview's stock
# checkpoint: 2 classes, 17 keypoints each -- the standard COCO-pose layout), while the
# actual trained weights only produce however many keypoint slots this dataset's real
# schema needs (typically 1, for this app's single "click_point" per object). Confirmed
# against a real training log: "Configured num_keypoints_per_class=[0, 17] does not
# match dataset keypoint metadata [1] ... Using dataset metadata as the source of
# truth" -- that correction lands on model_config, not on the postprocess object below.
# Sync it explicitly before exporting, or PostProcess._decode_keypoints_for_image raises
# "keypoints_i padded slot dimension (N) must equal num_keypoint_classes (...) *
# max_num_keypoints (...)" using the stale, pretrained-checkpoint schema instead of this
# dataset's real one.
if KEYPOINT_MODE:
    aligned_schema = list(getattr(model.model_config, "num_keypoints_per_class", []) or [])
    live_schema = list(getattr(model.model.postprocess, "num_keypoints_per_class", []) or [])
    if aligned_schema and aligned_schema != live_schema:
        print(f"Syncing postprocess keypoint schema {live_schema} -> {aligned_schema} before export")
        model.model.postprocess.num_keypoints_per_class = aligned_schema

# model.train() only leaves the LAST trained epoch's weights live in memory -- it never
# reloads the best-scoring checkpoint itself (RF-DETR only does that internally for its
# own optional trainer.test() step, which this notebook doesn't enable). With early
# stopping in particular, training keeps running EARLY_STOPPING_PATIENCE epochs past the
# best one before it actually stops, so the in-memory weights at this point are from a
# later, possibly-worse epoch than the best checkpoint already sitting on disk. Load
# that best checkpoint explicitly before tracing/exporting so best.onnx always reflects
# the model's actual peak, not whatever epoch training happened to stop at.
BEST_CKPT_PATH = Path("output") / "checkpoint_best_total.pth"
if BEST_CKPT_PATH.exists():
    from rfdetr.utilities.io import _safe_torch_load

    best_ckpt = _safe_torch_load(BEST_CKPT_PATH, trust=True)  # produced locally by this same run -- trusted
    raw_module.load_state_dict(best_ckpt["model"], strict=True)
    print(
        f"Loaded best checkpoint for export: {BEST_CKPT_PATH} "
        f"(epoch {best_ckpt.get('epoch')}, source={best_ckpt.get('best_total_source', 'unknown')})"
    )
else:
    print(f"{BEST_CKPT_PATH} not found -- exporting the last trained epoch's weights instead.")

raw_module.export()   # swap in export-safe submodule implementations -- see docstring above
raw_module.eval()
for p in raw_module.parameters():
    p.requires_grad = False

# Only patches this one PostProcess instance -- doesn't touch the class or any other
# instance, so normal predict() calls elsewhere in this notebook are unaffected.
model.model.postprocess._select_topk = types.MethodType(_select_topk_onnx_friendly, model.model.postprocess)
# Disables keypoint-uncertainty score fusion, which internally calls torch.nextafter
# (not exportable to ONNX at all) -- we only want the plain confidence score/keypoint
# xy this app actually uses, not the uncertainty-weighted refinement.
model.model.postprocess.trace_alpha = 0.0

wrapper = ExportWrapper(raw_module, model.model.postprocess, KEYPOINT_MODE, INPUT_SIZE)
wrapper.eval()
dummy_input = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE, device=next(raw_module.parameters()).device)

with torch.no_grad():
    torch.onnx.export(
        wrapper, dummy_input, "best.onnx",
        input_names=["images"], output_names=["output0"],
        opset_version=17,
        # The newer default exporter (dynamo=True) needs the onnxscript package (not
        # installed here) and can't trace this wrapper's predict()-adjacent code anyway
        # -- see this cell's markdown.
        dynamo=False,
    )
print("Exported best.onnx (keypoint_mode=", KEYPOINT_MODE, ")")

## 7. Sanity-check the export (optional)

Quick check that the exported graph actually produces plausible detections before downloading it — confirms the output shape and that a real training image's top-scoring row looks like a real detection (a sensible box, a confident score, a keypoint that falls inside that box), not noise. Safe to skip (or delete this cell) if you'd rather just download `best.onnx` and test it directly in the app — this only exists to catch a broken export earlier, not because the app needs it.


In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image

sess = ort.InferenceSession("best.onnx", providers=["CPUExecutionProvider"])
img = Image.open(next((RAW_DIR / "images").glob("*.png"))).convert("RGB").resize((INPUT_SIZE, INPUT_SIZE))
blob = (np.array(img).astype(np.float32) / 255.0).transpose(2, 0, 1)[None, ...]
output = sess.run(None, {"images": blob})[0]
print("output shape:", output.shape)  # (num_select, 6 or 8): x1,y1,x2,y2,score,label[,kx,ky]
top = output[np.argsort(-output[:, 4])[:5]]
print(top)


## 8. Download the trained model


In [ ]:
if ON_KAGGLE:
    # Kaggle captures every file left in /kaggle/working/ as this kernel's output --
    # no equivalent of Colab's interactive download; cv_training.py's poll/pull step
    # fetches it afterward via `kaggle kernels output`.
    print("On Kaggle: best.onnx left in /kaggle/working/ -- fetched by the app's Kaggle poll/pull step.")
else:
    from google.colab import files
    files.download("best.onnx")